In [1]:
# asyncpg: 비동기, httpx: 외부 API(http)비동기 호출
%pip install asyncpg httpx python-dotenv
%pip install requests


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# 테스트용 초기 state 만들기
from state import TravelState, UserInput, Place, make_initial_state
from constants.mocks import mock_user_input

initial_state = make_initial_state(mock_user_input)

In [4]:
# [노드] validate_input
# 사용자 입력 -> 좌표 변환, 식사 시간 계산 (전처리)
from nodes.validate_input import validate_input

validate_result = validate_input(initial_state)
print("validate_input 결과:", validate_result)


validate_input 결과: {'user_input': {'location': '해운대역', 'party_size': 2, 'party_type': '연인', 'genders': '혼성', 'age_group': '20대', 'duration': '당일', 'mood_preferences': ['활기찬', '힐링', '이색'], 'activity_preferences': ['카페', '게임/보드게임', '동물체험', '액티비티', '바다', '이색체험', '맛집'], 'dislike_keywords': ['PC', '술집'], 'trip_date': '2026-04-24', 'center_lat': 35.1636479638612, 'center_lng': 129.158897240251, 'search_radius_km': 1.5, 'final_keywords': ['카페', '게임/보드게임', '동물체험', '액티비티', '바다', '이색체험', '맛집']}, 'errors': [], 'warnings': [], 'step': 'validated'}


In [5]:
# [노드] fetch_candidates
# kakao Local API로 raw 후보 풀 수집 + PostgreSQL
from nodes.fetch_candidates import fetch_candidates

candidates_result = await fetch_candidates(validate_result)
print("candidates_result 결과:", candidates_result)


candidates_result 결과: {'user_input': {'location': '해운대역', 'party_size': 2, 'party_type': '연인', 'genders': '혼성', 'age_group': '20대', 'duration': '당일', 'mood_preferences': ['활기찬', '힐링', '이색'], 'activity_preferences': ['카페', '게임/보드게임', '동물체험', '액티비티', '바다', '이색체험', '맛집'], 'dislike_keywords': ['PC', '술집'], 'trip_date': '2026-04-24', 'center_lat': 35.1636479638612, 'center_lng': 129.158897240251, 'search_radius_km': 1.5, 'final_keywords': ['카페', '찻집', '보드게임카페', 'PC방', '오락실', '방탈출카페', 'VR체험', '동물카페', '고양이카페', '강아지카페', '라쿤카페', '양카페', '동물원', '수족관', '아쿠아리움', '짚라인', '루지', '케이블카', '번지점프', '놀이공원', '워터파크', '테마파크', '스포츠 시설', '해변', '해수욕장', '해안산책로', '바다', '이색체험', '팝업스토어', '글램핑', '캠핑장', '온천', '스파', '찜질방', '야시장', '드로잉카페', '한복', '한옥카페', '전통찻집', '갯벌체험', '체험', '맛집', '음식점']}, 'candidates': [{'id': '1538461447', 'name': '오프온', 'category': '음식점 > 카페', 'category_group_code': 'CE7', 'phone': '051-731-6058', 'address_name': '부산 해운대구 우동 517-12', 'road_address_name': '부산 해운대구 우동1로38번길 12', 'lat': 35.16526213188926

In [6]:
from nodes.filter_candidates import filter_candidates

filter_result = filter_candidates(candidates_result)
print("filter_result 결과:", filter_result)

filter_result 결과: {'filtered_candidates': [{'id': '12976118', 'name': '강릉송정해변막국수 부산분점', 'category': '음식점 > 한식 > 국수', 'category_group_code': 'FD6', 'phone': '051-746-0402', 'address_name': '부산 해운대구 중동 800-19', 'road_address_name': '부산 해운대구 좌동순환로446번길 15-6', 'lat': 35.162202531354, 'lng': 129.172602068256, 'place_url': 'http://place.map.kakao.com/12976118', 'tags': [], 'rating': 0.0, 'avg_stay_minutes': 60, 'open_hours': {}}, {'id': '59328169', 'name': '해운대 바다김밥', 'category': '음식점 > 분식', 'category_group_code': 'FD6', 'phone': '', 'address_name': '부산 해운대구 중동 1378-7', 'road_address_name': '부산 해운대구 중동1로 33-1', 'lat': 35.162539343706, 'lng': 129.163251812897, 'place_url': 'http://place.map.kakao.com/59328169', 'tags': [], 'rating': 0.0, 'avg_stay_minutes': 60, 'open_hours': {}}, {'id': '8379689', 'name': '바다마루전복죽', 'category': '음식점 > 한식 > 죽', 'category_group_code': 'FD6', 'phone': '051-746-0397', 'address_name': '부산 해운대구 중동 971', 'road_address_name': '부산 해운대구 달맞이길62번길 7', 'lat': 35.160490827

In [7]:
from nodes.enrich_candidates import enrich_candidates

test_state = {
    "filtered_candidates": filter_result["filtered_candidates"],
    "warnings": [],
}

enrich_result = await enrich_candidates(test_state)

print(f"⚠️ warnings: {enrich_result['warnings']}")
print(f"✅ step: {enrich_result['step']}")
print(f"📦 보강된 장소 수: {len(enrich_result['filtered_candidates'])}")
for p in enrich_result["filtered_candidates"]:
    print(f"""
이름: {p['name']}
분위기: {p.get('atmosphere')}
추천대상: {p.get('best_for')}
재방문의사: {p.get('revisit_intent')}
한줄요약: {p.get('summary')}
""")


⚠️ warnings: []
✅ step: enriched
📦 보강된 장소 수: 50

이름: 강릉송정해변막국수 부산분점
분위기: ['활기찬']
추천대상: ['친구', '가족']
재방문의사: high
한줄요약: 신선한 막국수와 수육을 제공합니다.


이름: 해운대 바다김밥
분위기: ['힐링']
추천대상: ['혼자', '친구']
재방문의사: medium
한줄요약: 다양한 김밥이 매력적인 곳입니다.


이름: 바다마루전복죽
분위기: ['따뜻한']
추천대상: ['혼자', '가족']
재방문의사: medium
한줄요약: 전복죽이 인기인 전통 식당입니다.


이름: 온천떡집
분위기: ['따뜻한', '조용한']
추천대상: ['혼자', '가족']
재방문의사: medium
한줄요약: 아산 온천시장에서 맛볼 수 있습니다.


이름: 해목
분위기: ['감성']
추천대상: ['연인', '혼자']
재방문의사: high
한줄요약: 일본 감성을 느낄 수 있는 식당입니다.


이름: 빨간떡볶이
분위기: ['활기찬']
추천대상: ['친구']
재방문의사: high
한줄요약: 매운 떡볶이를 즐길 수 있는 곳입니다.


이름: 타이가텐푸라
분위기: ['이색']
추천대상: ['친구', '가족']
재방문의사: medium
한줄요약: 오션뷰가 매력적인 텐푸라 전문점입니다.


이름: 고래사어묵 해운대점
분위기: ['활기찬']
추천대상: ['가족', '친구']
재방문의사: medium
한줄요약: 부산 기념품과 어묵을 즐길 수 있습니다.


이름: 오복돼지국밥
분위기: ['따뜻한']
추천대상: ['가족', '혼자']
재방문의사: high
한줄요약: 부산의 전통 돼지국밥을 제공합니다.


이름: 가야밀면
분위기: ['힙한']
추천대상: ['친구']
재방문의사: medium
한줄요약: 비빔밀면과 갈비만두가 인기입니다.


이름: 딤타오 해리단길점
분위기: ['이색']
추천대상: ['혼자', '친구']
재방문의사: medium
한줄요약: 딤섬과 디저트를 함께 즐길 수 있습니다.


이름: 금문
분위기: ['이

In [8]:
import time
from utils.enrich_with_llm import enrich_with_llm
from utils.search_naver_blogs import search_naver_blogs

# 블로그 수집 시간
start = time.time()
blog_data = await search_naver_blogs(filter_result["filtered_candidates"])
print(f"블로그 수집: {time.time() - start:.1f}초")

# LLM 시간
start = time.time()
llm_results = await enrich_with_llm(blog_data)
print(f"LLM 호출: {time.time() - start:.1f}초")

블로그 수집: 11.3초
LLM 호출: 30.5초


In [9]:
# # LangGraph 그래프 빌드
# from langgraph.graph import StateGraph, START, END
#
# graph_builder = StateGraph(TravelState)
#
# # 노드 등록
# graph_builder.add_node("validate_input", validate_input)
# graph_builder.add_node("fetch_candidates", fetch_candidates)
# graph_builder.add_node("filter_candidates", filter_candidates)
# graph_builder.add_node("enrich_candidates", enrich_candidates)
#
# # 엣지 (직선 연결)
# graph_builder.add_edge(START, "validate_input")
# graph_builder.add_edge("validate_input", "fetch_candidates")
# graph_builder.add_edge("fetch_candidates", "filter_candidates")
# graph_builder.add_edge("filter_candidates", "enrich_candidates")
# graph_builder.add_edge("enrich_candidates", END)
#
# graph = graph_builder.compile()

In [10]:
# # 그래프 실행
# state_v1 = await graph.ainvoke(initial_state)
#
# print(f"📍 위치: {state_v1['user_input']['location']}")
# print(f"📌 좌표: ({state_v1['user_input']['center_lat']}, {state_v1['user_input']['center_lng']})")
# print(f"🔍 Kakao raw 후보: {len(state_v1['candidates'])}개")
# print(f"⚠️  warnings: {state_v1['warnings']}")
# print(f"❌ errors: {state_v1['errors']}")
# print(f"✅ step: {state_v1['step']}\n")
#
# for p in filter_result["filtered_candidates"][:5]:
#     print(
#         f"  [{p.get('category_group_code', '기타')}] {p['name']} - "
#         f"{p.get('road_address_name') or p.get('address_name', '')}"
#     )

In [11]:
# from IPython.display import Image, display
# display(Image(graph.get_graph().draw_mermaid_png()))